# Albania Urban Air Quality Early Warning Dashboard

This JupyterLab dashboard uses the daily analytical layer of the project to compare cities, inspect risk patterns, and review high-pollution days.


In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, Markdown


In [2]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

daily = pd.read_csv(PROCESSED_DIR / "albania_air_quality_daily.csv", parse_dates=["date"])
summary = pd.read_csv(PROCESSED_DIR / "city_risk_summary.csv")
latest_snapshot = pd.read_csv(PROCESSED_DIR / "latest_city_snapshot.csv", parse_dates=["date"])
events = pd.read_csv(PROCESSED_DIR / "pollution_episode_days.csv", parse_dates=["date"])

summary_lookup = summary.set_index("city")
latest_lookup = latest_snapshot.set_index("city")

display(Markdown(
    f"Loaded **{len(daily):,}** daily records across **{daily['city'].nunique()}** Albanian cities."
))
latest_snapshot[["city", "date", "european_aqi_max", "aqi_label"]].head()


Loaded **6,792** daily records across **8** Albanian cities.

,city,date,european_aqi_max,aqi_label
0,Durres,2026-04-28,53,Moderate
1,Shkoder,2026-04-28,49,Moderate
2,Tirane,2026-04-28,48,Moderate
3,Berat,2026-04-28,47,Moderate
4,Fier,2026-04-28,47,Moderate


In [3]:
aqi_colors = {
    "Good": "#1f77b4",
    "Fair": "#2ca02c",
    "Moderate": "#bcbd22",
    "Poor": "#ff7f0e",
    "Very Poor": "#d62728",
    "Extremely Poor": "#7f0000",
    "Unknown": "#7f7f7f",
}

metric_options = {
    "Daily Max AQI": "european_aqi_max",
    "Daily Mean AQI": "european_aqi_mean",
    "Daily Mean PM2.5": "pm2_5_mean",
    "Daily Max PM2.5": "pm2_5_max",
    "Daily Mean PM10": "pm10_mean",
    "Daily Max PM10": "pm10_max",
    "Daily Mean NO2": "nitrogen_dioxide_mean",
    "Daily Max NO2": "nitrogen_dioxide_max",
}

city_widget = widgets.Dropdown(
    options=sorted(daily["city"].unique()),
    value="Tirane",
    description="City:",
    layout=widgets.Layout(width="240px"),
)

metric_widget = widgets.Dropdown(
    options=list(metric_options.keys()),
    value="Daily Max AQI",
    description="Metric:",
    layout=widgets.Layout(width="260px"),
)

window_widget = widgets.IntSlider(
    value=120,
    min=30,
    max=365,
    step=15,
    description="Window:",
    continuous_update=False,
)

top_n_widget = widgets.IntSlider(
    value=10,
    min=5,
    max=20,
    step=1,
    description="Top Days:",
    continuous_update=False,
)

output = widgets.Output()


In [4]:
def render_dashboard(*_):
    output.clear_output(wait=True)

    city = city_widget.value
    metric_label = metric_widget.value
    metric_col = metric_options[metric_label]
    window_days = window_widget.value
    top_n = top_n_widget.value

    city_daily = daily[daily["city"] == city].copy().sort_values("date")
    city_window = city_daily.tail(window_days).copy()
    city_events = events[events["city"] == city].copy().sort_values("date", ascending=False).head(top_n)

    latest_city = latest_lookup.loc[city]
    summary_city = summary_lookup.loc[city]

    compare_df = latest_snapshot[["city", metric_col, "aqi_label"]].copy()
    compare_df = compare_df.sort_values(metric_col, ascending=False)

    trend_fig = px.line(
        city_window,
        x="date",
        y=metric_col,
        template="plotly_white",
        title=f"{metric_label} - {city} (last {window_days} days)",
    )
    trend_fig.update_traces(line=dict(width=2, color="#1f77b4"))
    trend_fig.update_layout(height=420, xaxis_title="Date", yaxis_title=metric_label)

    compare_fig = px.bar(
        compare_df,
        x="city",
        y=metric_col,
        color="aqi_label",
        color_discrete_map=aqi_colors,
        template="plotly_white",
        title=f"Latest city comparison - {metric_label}",
    )
    compare_fig.update_layout(height=420, xaxis_title="City", yaxis_title=metric_label, legend_title_text="AQI label")

    top_days = city_daily.nlargest(top_n, "european_aqi_max")[["date", "european_aqi_max", "aqi_label", "pm2_5_max", "pm10_max", "dominant_pollutant"]]
    top_fig = px.bar(
        top_days.sort_values("european_aqi_max"),
        x="european_aqi_max",
        y=top_days.sort_values("european_aqi_max")["date"].dt.strftime("%Y-%m-%d"),
        color="aqi_label",
        color_discrete_map=aqi_colors,
        orientation="h",
        template="plotly_white",
        title=f"Top {top_n} AQI days - {city}",
    )
    top_fig.update_layout(height=420, xaxis_title="Daily max AQI", yaxis_title="Date", legend_title_text="AQI label")

    with output:
        display(Markdown(
            f"## Operational Snapshot - {city}\n"
            f"- Latest available date: **{pd.to_datetime(latest_city['date']).date()}**\n"
            f"- Latest daily max AQI: **{latest_city['european_aqi_max']:.1f}** ({latest_city['aqi_label']})\n"
            f"- Average PM2.5 across the study window: **{summary_city['avg_pm2_5']:.2f}** ug/m3\n"
            f"- Pollution episode days: **{int(summary_city['episode_days'])}** of **{int(summary_city['days_observed'])}**\n"
            f"- High-risk days: **{int(summary_city['high_risk_days'])}**\n"
            f"- Highest observed AQI: **{summary_city['max_aqi']:.1f}**"
        ))
        display(compare_fig)
        display(trend_fig)
        display(top_fig)
        display(Markdown(f"### Recent episode log - {city}"))
        if city_events.empty:
            display(Markdown("No episode days were detected for the current filter."))
        else:
            display(city_events[["date", "aqi_label", "european_aqi_max", "pm2_5_max", "pm10_max", "dominant_pollutant"]].reset_index(drop=True))


for widget in [city_widget, metric_widget, window_widget, top_n_widget]:
    widget.observe(render_dashboard, names="value")

display(widgets.HBox([city_widget, metric_widget, window_widget, top_n_widget]))
display(output)
render_dashboard()


Output()

In [ ]:
summary.sort_values(["high_risk_days", "episode_days", "avg_aqi"], ascending=[False, False, False]).reset_index(drop=True)
